In [13]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split


In [7]:
data = pd.read_csv('Modelling_data.csv')

In [8]:
data.sample(5)

,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
892,50893,0,0.0,Phone,1,17.0,card,Male,3.0,3.0,Phone,5,Divorced,1,0,14.0,1.0,1.0,4.0,120.99
1587,51588,0,14.0,Phone,1,25.0,card,Male,0.0,3.0,Laptop & Accessory,5,Married,3,0,16.0,0.0,1.0,0.0,130.72
2736,52737,0,11.0,Phone,3,7.0,card,Male,0.0,4.0,Phone,5,Single,5,0,14.0,1.0,1.0,0.0,128.58
3585,53586,0,5.0,Phone,1,18.0,card,Male,4.0,4.0,Laptop & Accessory,5,Divorced,3,1,13.0,1.0,2.0,2.0,193.33
2427,52428,0,0.0,Phone,1,26.0,card,Male,3.0,3.0,Laptop & Accessory,3,Single,1,0,14.0,1.0,1.0,2.0,122.72


In [35]:
cat_cols = X.select_dtypes(include=['object']).columns
num_cols = X.select_dtypes(include=['int64','float64']).columns

cat_cols, num_cols

(Index(['MaritalStatus'], dtype='object'),
 Index(['Tenure', 'Complain', 'NumberOfAddress', 'CashbackAmount',
        'WarehouseToHome', 'DaySinceLastOrder', 'SatisfactionScore',
        'CityTier'],
       dtype='object'))

In [9]:
xgbpipe = Pipeline(steps=[
    ('model', XGBClassifier(
        n_estimators=2000,
        learning_rate=0.08,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric='logloss'
    ))
])

In [33]:
X = data[['Tenure', 'Complain', 'NumberOfAddress', 
          'CashbackAmount', 'MaritalStatus', 'WarehouseToHome', 'DaySinceLastOrder', 'SatisfactionScore', 'CityTier']]
y = data['Churn']

In [36]:
from sklearn.preprocessing import LabelEncoder

label_encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    label_encoders[col] = le

C:\Users\USER\AppData\Local\Temp\ipykernel_18060\1911248483.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = le.fit_transform(X[col])


In [37]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [38]:
xgbpipe.fit(X_train,y_train)

,steps,"[('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8


In [39]:
y_pred = xgbpipe.predict(X_test)

In [40]:
from sklearn.metrics import classification_report, accuracy_score

print("Accuracy:", accuracy_score(y_test, y_pred)*100)
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 98.3126110124334

Classification Report:
               precision    recall  f1-score   support

           0       0.98      1.00      0.99       941
           1       0.99      0.91      0.95       185

    accuracy                           0.98      1126
   macro avg       0.99      0.95      0.97      1126
weighted avg       0.98      0.98      0.98      1126



In [41]:
import joblib
joblib.dump(xgbpipe, "xgb_churn_model.pkl")

['xgb_churn_model.pkl']

In [42]:
X

,Tenure,Complain,NumberOfAddress,CashbackAmount,MaritalStatus,WarehouseToHome,DaySinceLastOrder,SatisfactionScore,CityTier
0,4.0,1,9,159.93,2,6.0,5.0,2,3
1,0.0,1,7,120.90,2,8.0,0.0,3,1
2,0.0,1,6,120.28,2,30.0,3.0,3,1
3,0.0,0,8,134.07,2,15.0,3.0,5,3
4,0.0,0,3,129.60,2,12.0,3.0,5,1
...,...,...,...,...,...,...,...,...,...
5625,10.0,0,6,150.71,1,30.0,4.0,1,1
5626,13.0,0,6,224.91,1,13.0,0.0,5,1
5627,1.0,1,3,186.42,1,11.0,4.0,4,1
5628,23.0,0,4,178.90,1,9.0,9.0,4,3
